**Workshop notebooks:** **01 — Matching & Loading (you are here)**&nbsp;·&nbsp;[02 — Overlay & Enrichment](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/02_overlay_enrichment.ipynb)&nbsp;·&nbsp;[03 — Bridging](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/03_bridging.ipynb)&nbsp;·&nbsp;[04 — Disease Modules (optional)](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/04_disease_modules_optional.ipynb)

# 01 — Matching & Loading

**Network Medicine Workshop · Kidney Disease · Part 1 of 3 (+ optional Part 4)**

This notebook handles the unglamorous-but-essential part of network medicine: getting *your* messy omics lists onto standardized IDs that match the node IDs used in the networks, and loading + sanity-checking those networks.

You're bringing **three independent datasets**, each from a different omics layer:
- **Transcripts** — DEGs from spatial transcriptomics
- **Proteins** — DE proteins from proteomics
- **Metabolites** — DE metabolites from metabolomics

Each gets matched to the ID system used by its corresponding network: transcripts and proteins both resolve to **NCBI Gene IDs** (transcripts → transcriptome network, proteins → PPI network), metabolites resolve to **KEGG Compound IDs** (→ metabolite network).

**What you'll do here (~15–20 min):**
1. Load all three DE lists
2. Batch-map gene/protein symbols → NCBI Gene IDs (with a fuzzy fallback for typos/old symbols)
3. Batch-map metabolite names → KEGG Compound IDs (local KEGG dictionary + vectorized fuzzy matching)
4. Load each network from your Drive `networks/` folder and print basic stats
5. Save everything to `processed/` so later notebooks can pick it up

**You will need, in your Drive workshop folder:**
- `data/DEGs.csv` — needs a `gene_symbol` column (transcripts)
- `data/DE_proteins.csv` — needs a `protein_symbol` column (the gene symbol encoding each protein; extra columns like fold-change kept)
- `data/DE_metabolites.csv` — needs a `metabolite_name` column
- `networks/ppi_network.csv` — edges `source,target[,weight]`, **NCBI Gene IDs**
- `networks/transcriptome_network.csv` — same format, **NCBI Gene IDs**
- `networks/metabolite_network.csv` — same format, **KEGG Compound IDs**

If your column names differ, just rename them below in the config cell — nothing else needs to change.


## Setup

In [63]:
# Install what we need. mygene = batch gene ID mapping, rapidfuzz = fast fuzzy string matching.
!pip install -q mygene rapidfuzz networkx requests tqdm pandas



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [64]:
from google.colab import drive
drive.mount('/content/drive')


ModuleNotFoundError: No module named 'google'

In [66]:
# ---- CONFIG ----
import os

GITHUB_REPO = "marlene-grabner/NetworkMedicine_Workshop"
GITHUB_BRANCH = "main"
GITHUB_RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}"

# Drive is just persistent scratch space between notebooks - nothing needs to be
# manually placed here, Step 0 below downloads everything into it automatically.
BASE_DIR = "/content/drive/MyDrive/network_medicine_workshop"

DATA_DIR = os.path.join(BASE_DIR, "data")
NET_DIR = os.path.join(BASE_DIR, "networks")
PROC_DIR = os.path.join(BASE_DIR, "processed")
for d in (DATA_DIR, NET_DIR, PROC_DIR):
    os.makedirs(d, exist_ok=True)

DEG_FILE = os.path.join(DATA_DIR, "DEGs.csv")               # transcripts
PROTEIN_FILE = os.path.join(DATA_DIR, "DE_proteins.csv")    # proteins
METAB_FILE = os.path.join(DATA_DIR, "DE_metabolites.csv")   # metabolites

GENE_SYMBOL_COL = "gene_symbol"
PROTEIN_SYMBOL_COL = "protein_symbol"
METAB_NAME_COL = "metabolite_name"

# network layer -> (filename, which dataset maps onto it)
NETWORK_FILES = {
    "transcriptome": "transcriptomes_coexpression_kidney.tsv",   # matched against DEGs (transcripts), NCBI Gene IDs
    "ppi":           "ppi.tsv",             # matched against DE proteins, NCBI Gene IDs
    "metabolite":    "metabolite_network.tsv",      # matched against DE metabolites, KEGG Compound IDs
}
BRIDGE_FILE = "metabolite_gene_bridge.csv"           # used later, in Notebook 3

print("Base dir:", BASE_DIR)

OSError: [Errno 30] Read-only file system: '/content'

## Step 0 — Fetch data & networks from GitHub

Everyone runs this. It downloads the workshop's data and network files straight from the public repo into your Drive — nothing to upload by hand. Already-downloaded files are skipped on re-runs (delete a file if you want it re-fetched, e.g. because it changed upstream).

In [ ]:
import requests

FILES_TO_FETCH = {
    f"{GITHUB_RAW_BASE}/data/DEGs.csv": DEG_FILE,
    f"{GITHUB_RAW_BASE}/data/DE_proteins.csv": PROTEIN_FILE,
    f"{GITHUB_RAW_BASE}/data/DE_metabolites.csv": METAB_FILE,
    **{
        f"{GITHUB_RAW_BASE}/networks/{fname}": os.path.join(NET_DIR, fname)
        for fname in NETWORK_FILES.values()
    },
    f"{GITHUB_RAW_BASE}/networks/{BRIDGE_FILE}": os.path.join(NET_DIR, BRIDGE_FILE),
}

for url, dest in FILES_TO_FETCH.items():
    if os.path.exists(dest):
        print(f"already have {os.path.basename(dest)} (delete it to re-download)")
        continue
    r = requests.get(url, timeout=60)
    if r.status_code == 200:
        with open(dest, "wb") as f:
            f.write(r.content)
        print(f"downloaded {os.path.basename(dest)}  ({len(r.content):,} bytes)")
    else:
        print(f"[warn] could not fetch {url} (status {r.status_code}) — "
              f"check GITHUB_REPO above and that the file exists in the repo yet")

SyntaxError: invalid syntax (3450606112.py, line 3)

## See what just landed on your Drive

Two ways to look at the downlaod we just did without leaving this tab:

- **Colab's file browser**: click the 📁 folder icon on the far left sidebar → `drive` → `MyDrive` → `network_medicine_workshop`. Double-click any CSV/TSV there and Colab opens a proper preview pane.
- Or just look at the printout below, which lists exactly what's there and how big each file is.

(You can also find the same folder by opening `drive.google.com` in another tab if you'd rather — it's a completely normal Drive folder, nothing Colab-specific.)


In [67]:
print("Files now sitting in your Drive, under network_medicine_workshop/:\n")
for folder, label in [(DATA_DIR, "data/"), (NET_DIR, "networks/")]:
    print(label)
    for fname in sorted(os.listdir(folder)):
        size_kb = os.path.getsize(os.path.join(folder, fname)) / 1024
        print(f"  {fname}  ({size_kb:.1f} KB)")
    print()


Files now sitting in your Drive, under network_medicine_workshop/:

data/


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/network_medicine_workshop/data'

In [ ]:
# A raw peek at one of them, exactly as downloaded - before any Python touches it
print("First few lines of DEGs.csv, unprocessed:\n")
!head -n 5 {DEG_FILE}


## Step 1 — Load your three lists

Nothing fancy here, just load and take a quick look at each. This is also your chance to catch obvious problems (wrong column names, encoding issues, duplicate entries) before they propagate downstream.


In [47]:
import pandas as pd

degs = pd.read_csv(DEG_FILE)
proteins = pd.read_csv(PROTEIN_FILE)
with open(METAB_FILE, "r") as f:
    lines = [line.strip() for line in f if line.strip()]
header = lines[0].split(",")
metabolite_list = lines[1:]
metabs = pd.DataFrame(metabolite_list, columns=header).drop_duplicates()

# Basic hygiene: strip whitespace, drop exact duplicates
degs[GENE_SYMBOL_COL] = degs[GENE_SYMBOL_COL].astype(str).str.strip()
proteins[PROTEIN_SYMBOL_COL] = proteins[PROTEIN_SYMBOL_COL].astype(str).str.strip()
metabs[METAB_NAME_COL] = metabs[METAB_NAME_COL].astype(str).str.strip()

degs = degs.drop_duplicates(subset=GENE_SYMBOL_COL).reset_index(drop=True)
proteins = proteins.drop_duplicates(subset=PROTEIN_SYMBOL_COL).reset_index(drop=True)
metabs = metabs.drop_duplicates(subset=METAB_NAME_COL).reset_index(drop=True)

print(f"Transcripts (DEGs):    {len(degs)} unique gene symbols")
print(f"Proteins (proteomics): {len(proteins)} unique protein/gene symbols")
print(f"Metabolites:           {len(metabs)} unique metabolite names")
degs.head()


Transcripts (DEGs):    438 unique gene symbols
Proteins (proteomics): 8 unique protein/gene symbols
Metabolites:           56 unique metabolite names


,gene_symbol
0,EME1
1,FEN1
2,NEIL3
3,POLQ
4,CHEK1


In [48]:
proteins.head()


,protein_symbol
0,EME1
1,FEN1
2,NEIL3
3,POLQ
4,CHEK1


In [49]:
metabs.head()


,metabolite_name
0,Phosphatidylserine
1,L-1-Lysophosphatidylethanolamine
2,N-Acylglycine
3,Galactosylceramide
4,Phosphatidate


## Step 2 — Gene/protein symbol → NCBI Gene ID

Transcripts and proteins are both matched the same way, since both ultimately resolve to a gene identifier — we just do it as **one batch call per dataset** rather than looping symbol-by-symbol. `mygene.info`'s `querymany` accepts up to ~1000 identifiers per request and resolves current symbols *and* aliases/old names in a single pass. This is the single biggest speed difference vs. the naive approach.

Anything still unmatched after that gets a fuzzy-matching pass against the pool of symbols the batch call *did* resolve, so a typo like `"HAVCR-1"` still finds `HAVCR1`.

We wrap this in one reusable function and call it twice — once for transcripts, once for proteins — so both datasets get identical treatment.


In [11]:
import mygene
from rapidfuzz import process, fuzz

mg = mygene.MyGeneInfo()

def match_symbols_to_ncbi(symbols, label):
    """Batch-map a list of gene/protein symbols to NCBI Gene IDs, with fuzzy fallback."""
    result = mg.querymany(
        symbols, scopes="symbol,alias", fields="entrezgene,symbol",
        species="human", returnall=True,
    )

    matched_rows = []
    seen = set()
    for hit in result["out"]:
        if "entrezgene" in hit and hit["query"] not in seen:
            matched_rows.append({
                "query": hit["query"],
                "ncbi_gene_id": str(hit["entrezgene"]),
                "matched_symbol": hit.get("symbol", ""),
                "match_type": "exact/alias",
                "match_score": 100,
            })
            seen.add(hit["query"])

    matches = pd.DataFrame(matched_rows).drop_duplicates(subset="query")
    missing = sorted(set(symbols) - set(matches["query"]))
    print(f"[{label}] matched directly: {len(matches)} / {len(symbols)}  |  unmatched: {len(missing)}")

    if missing:
        reference_symbols = matches["matched_symbol"].tolist()
        fuzzy_rows, still_missing = [], []
        for q in missing:
            best = process.extractOne(q, reference_symbols, scorer=fuzz.WRatio)
            if best and best[1] >= 85:
                row = matches.loc[matches["matched_symbol"] == best[0]].iloc[0]
                fuzzy_rows.append({
                    "query": q, "ncbi_gene_id": row["ncbi_gene_id"],
                    "matched_symbol": best[0], "match_type": "fuzzy", "match_score": best[1],
                })
            else:
                still_missing.append(q)
        matches = pd.concat([matches, pd.DataFrame(fuzzy_rows)], ignore_index=True)
        print(f"[{label}] recovered via fuzzy matching (score>=85): {len(fuzzy_rows)}  |  "
              f"genuinely unmatched: {len(still_missing)}")
        if still_missing:
            print(f"[{label}] unmatched examples:", still_missing[:15])

    return matches

gene_matches = match_symbols_to_ncbi(degs[GENE_SYMBOL_COL].tolist(), "transcripts")
protein_matches = match_symbols_to_ncbi(proteins[PROTEIN_SYMBOL_COL].tolist(), "proteins")


25 input query terms found dup hits:	[('POLQ', 2), ('MCM2', 2), ('ORC1', 2), ('MAP2', 2), ('DSC1', 2), ('DSC2', 2), ('MET', 2), ('FHOD3',


[transcripts] matched directly: 438 / 438  |  unmatched: 0


1 input query terms found dup hits:	[('POLQ', 2)]


[proteins] matched directly: 8 / 8  |  unmatched: 0


In [50]:
gene_matches.sort_values("match_score").head(10)   # eyeball the shakiest transcript matches


,query,ncbi_gene_id,matched_symbol,match_type,match_score
0,EME1,146956,EME1,exact/alias,100
298,HLA-C,3107,HLA-C,exact/alias,100
297,HLA-B,3106,HLA-B,exact/alias,100
296,HLA-A,3105,HLA-A,exact/alias,100
295,B2M,567,B2M,exact/alias,100
294,MCAM,4162,MCAM,exact/alias,100
293,JAM3,83700,JAM3,exact/alias,100
292,VCAM1,7412,VCAM1,exact/alias,100
291,ICAM1,3383,ICAM1,exact/alias,100
290,APOBEC3A,200315,APOBEC3A,exact/alias,100


In [51]:
protein_matches.sort_values("match_score").head(10)   # eyeball the shakiest protein matches


,query,ncbi_gene_id,matched_symbol,match_type,match_score
0,EME1,146956,EME1,exact/alias,100
1,FEN1,2237,FEN1,exact/alias,100
2,NEIL3,55247,NEIL3,exact/alias,100
3,POLQ,51426,POLK,exact/alias,100
4,CHEK1,1111,CHEK1,exact/alias,100
5,FBXO5,26271,FBXO5,exact/alias,100
6,H2AFX,3014,H2AX,exact/alias,100
7,UHRF1,29128,UHRF1,exact/alias,100


In [52]:
degs_matched = degs.merge(
    gene_matches[["query", "ncbi_gene_id", "matched_symbol", "match_type", "match_score"]],
    left_on=GENE_SYMBOL_COL, right_on="query", how="left"
).drop(columns="query")

proteins_matched = proteins.merge(
    protein_matches[["query", "ncbi_gene_id", "matched_symbol", "match_type", "match_score"]],
    left_on=PROTEIN_SYMBOL_COL, right_on="query", how="left"
).drop(columns="query")

print(f"Transcripts: {degs_matched['ncbi_gene_id'].notna().sum()} / {len(degs_matched)} matched")
print(f"Proteins:    {proteins_matched['ncbi_gene_id'].notna().sum()} / {len(proteins_matched)} matched")


Transcripts: 438 / 438 matched
Proteins:    8 / 8 matched


## Step 3 — Metabolite name → KEGG Compound ID

Same philosophy, different resource: don't hit an API per metabolite. Instead we download the **full KEGG compound list once** (~19k entries, one request) and build a local name/synonym lookup. Exact matches resolve instantly; everything else goes through a single vectorized fuzzy-matching pass (`rapidfuzz.process.cdist`), which compares all your unmatched names against all KEGG names in one shot instead of nested loops.

Matches are scored so you can see, at a glance, which ones need a human to double check — this is the realistic way to handle "someone else's" metabolite naming conventions (brand names, plurals, stereochemistry prefixes, etc.).


In [53]:
import requests

resp = requests.get("https://rest.kegg.jp/list/compound", timeout=60)
resp.raise_for_status()

kegg_id_to_names = {}
for line in resp.text.strip().split("\n"):
    cid, names = line.split("\t")
    cid = cid.replace("cpd:", "").strip()
    kegg_id_to_names[cid] = [n.strip() for n in names.split(";")]

print(f"Loaded {len(kegg_id_to_names)} KEGG compounds")

# Flat lookup: lowercase name -> KEGG ID (first name wins if there's a collision)
name_to_kegg = {}
for cid, names in kegg_id_to_names.items():
    for n in names:
        name_to_kegg.setdefault(n.lower(), cid)

all_kegg_names = list(name_to_kegg.keys())


Loaded 19632 KEGG compounds


In [54]:
metab_names = metabs[METAB_NAME_COL].tolist()

exact_rows, unmatched_metabs = [], []
for name in metab_names:
    key = name.lower()
    if key in name_to_kegg:
        exact_rows.append({
            "query": name, "kegg_id": name_to_kegg[key],
            "matched_name": name, "match_type": "exact", "match_score": 100
        })
    else:
        unmatched_metabs.append(name)

print(f"Exact matches: {len(exact_rows)} / {len(metab_names)}")
print(f"Sent to fuzzy matching: {len(unmatched_metabs)}")


Exact matches: 55 / 56
Sent to fuzzy matching: 1


In [41]:
fuzzy_rows = []
if unmatched_metabs:
    import numpy as np
    # cdist = compute a full similarity matrix in one vectorized call (fast, C-backed)
    scores = process.cdist(unmatched_metabs, all_kegg_names, scorer=fuzz.WRatio, workers=-1)
    best_idx = scores.argmax(axis=1)
    best_scores = scores.max(axis=1)

    for name, idx, score in zip(unmatched_metabs, best_idx, best_scores):
        matched_name = all_kegg_names[idx]
        fuzzy_rows.append({
            "query": name, "kegg_id": name_to_kegg[matched_name],
            "matched_name": matched_name, "match_type": "fuzzy", "match_score": round(float(score), 1),
        })

metab_matches = pd.DataFrame(exact_rows + fuzzy_rows)

# Human-in-the-loop bands: >=90 auto-accept, 70-89 review, <70 likely wrong
metab_matches["review_flag"] = pd.cut(
    metab_matches["match_score"], bins=[0, 69.999, 89.999, 100],
    labels=["reject/manual", "review", "auto-accept"]
)
print(metab_matches["review_flag"].value_counts())
metab_matches.sort_values("match_score").head(15)


review_flag
auto-accept      56
reject/manual     0
review            0
Name: count, dtype: int64


,query,kegg_id,matched_name,match_type,match_score,review_flag
55,Orotidine 5-phosphate,C01103,orotidine 5'-phosphate,fuzzy,93.0,auto-accept
30,N-Acetyl-D-glutamate,C22611,N-Acetyl-D-glutamate,exact,100.0,auto-accept
31,Monoacylglycerol,C01885,Monoacylglycerol,exact,100.0,auto-accept
32,1-Acyl-sn-glycerol 3-phosphate,C00681,1-Acyl-sn-glycerol 3-phosphate,exact,100.0,auto-accept
33,gamma-L-Glutamyl-L-cysteine,C00669,gamma-L-Glutamyl-L-cysteine,exact,100.0,auto-accept
34,Docosanedioate,C19625,Docosanedioate,exact,100.0,auto-accept
35,D-Glucuronate 1-phosphate,C05385,D-Glucuronate 1-phosphate,exact,100.0,auto-accept
36,"5-(2'-Carboxyethyl)-4,6-dihydroxypicolinate",C05655,"5-(2'-Carboxyethyl)-4,6-dihydroxypicolinate",exact,100.0,auto-accept
37,2-Ketoglutaric acid,C00026,2-Ketoglutaric acid,exact,100.0,auto-accept
38,D-Gluconic acid,C00257,D-Gluconic acid,exact,100.0,auto-accept


In [55]:
# --- Manual review step ---
# Anything flagged "review" or "reject/manual" below is worth a human glance.
# Edit the dict below to fix specific entries, then re-run this cell.
manual_overrides = {
    # "your messy name here": "C00031",
}

for name, cid in manual_overrides.items():
    mask = metab_matches["query"] == name
    metab_matches.loc[mask, ["kegg_id", "match_type", "match_score", "review_flag"]] = [cid, "manual", 100, "auto-accept"]

metab_matches[metab_matches["review_flag"] != "auto-accept"]


,query,kegg_id,matched_name,match_type,match_score,review_flag


In [43]:
metabs_matched = metabs.merge(
    metab_matches[["query", "kegg_id", "match_type", "match_score", "review_flag"]],
    left_on=METAB_NAME_COL, right_on="query", how="left"
).drop(columns="query")

print(f"{metabs_matched['kegg_id'].notna().sum()} / {len(metabs_matched)} metabolites matched to a KEGG ID")
metabs_matched.head()


56 / 56 metabolites matched to a KEGG ID


,metabolite_name,kegg_id,match_type,match_score,review_flag
0,Phosphatidylserine,C02737,exact,100.0,auto-accept
1,L-1-Lysophosphatidylethanolamine,C05973,exact,100.0,auto-accept
2,N-Acylglycine,C02055,exact,100.0,auto-accept
3,Galactosylceramide,C02686,exact,100.0,auto-accept
4,Phosphatidate,C00416,exact,100.0,auto-accept


## Step 4 — Load & screen the networks

Just loading edge lists and reporting the numbers every network-medicine analysis should start with: how many nodes, how many edges, how dense, how many connected components, and how big the largest one is. At up to ~20k nodes these are all cheap (linear-time) operations in `networkx` — the thing to avoid later is anything that scales quadratically (e.g. all-pairs shortest paths), which we'll sidestep in later notebooks.


In [58]:
import networkx as nx
import time

graphs = {}
for layer, fname in NETWORK_FILES.items():
    path = os.path.join(NET_DIR, fname)
    G = nx.read_edgelist(path)
    
    t0 = time.time()
    """
    edges = pd.read_csv(path)
    edges.columns = [c.strip().lower() for c in edges.columns]
    assert {"source", "target"}.issubset(edges.columns), f"{fname} needs 'source' and 'target' columns"

    if "weight" in edges.columns:
        G = nx.from_pandas_edgelist(edges, "source", "target", edge_attr="weight")
    else:
        G = nx.from_pandas_edgelist(edges, "source", "target")

    # Node IDs from CSV often come in as int64/float64/str inconsistently - normalize to str
    G = nx.relabel_nodes(G, {n: str(n) for n in G.nodes()})
    """
    graphs[layer] = G
    n_cc = nx.number_connected_components(G)
    largest_cc = len(max(nx.connected_components(G), key=len))
    print(f"[{layer}] {fname}")
    print(f"   nodes={G.number_of_nodes():,}  edges={G.number_of_edges():,}  "
          f"density={nx.density(G):.5f}  components={n_cc}  largest_cc={largest_cc:,} "
          f"({largest_cc/G.number_of_nodes():.1%} of nodes)")
    print(f"   loaded in {time.time()-t0:.1f}s\n")


[transcriptome] coex_KDN_NCBI.tsv
   nodes=13,264  edges=545,988  density=0.00621  components=6  largest_cc=13,247 (99.9% of nodes)
   loaded in 0.1s

[ppi] ppi.tsv
   nodes=16,569  edges=292,671  density=0.00213  components=1  largest_cc=16,569 (100.0% of nodes)
   loaded in 0.1s

[metabolite] metabolite_network.tsv
   nodes=2,403  edges=5,917  density=0.00205  components=116  largest_cc=2,133 (88.8% of nodes)
   loaded in 0.0s



**What to look for:** a healthy PPI/co-expression network usually has one dominant connected component covering the large majority of nodes — if your largest component is small relative to the total, either the network is unusually fragmented or something's off in the edge list (e.g. duplicate ID systems mixed together). Worth flagging to the group live if it looks off.


## Step 5 — Save everything for the next notebooks

In [59]:
import pickle

for layer, G in graphs.items():
    with open(os.path.join(PROC_DIR, f"graph_{layer}.pkl"), "wb") as f:
        pickle.dump(G, f)

degs_matched.to_csv(os.path.join(PROC_DIR, "degs_matched.csv"), index=False)
proteins_matched.to_csv(os.path.join(PROC_DIR, "proteins_matched.csv"), index=False)
metabs_matched.to_csv(os.path.join(PROC_DIR, "metabolites_matched.csv"), index=False)

print("Saved to", PROC_DIR)
print(os.listdir(PROC_DIR))


Saved to ./processed
['graph_metabolite.pkl', 'degs_matched.csv', 'enrichment_transcriptome_connected_module.csv', 'bridge_module_enrichment.csv', 'metabolites_matched.csv', 'enrichment_transcriptome_full_de_list.csv', 's_ab_comparison.png', 'enrichment_ppi_full_de_list.csv', 'graph_transcriptome.pkl', 'bridge_module.png', 'bridge_module.pkl', 'connectivity_test.png', 'modules.pkl', 'proteins_matched.csv', 'graph_ppi.pkl']


---
**Next:** open `02_overlay_enrichment.ipynb` — it picks up exactly where this notebook left off. There's also an **optional Notebook 4** (`04_disease_modules_optional.ipynb`) for a deeper dive into module significance and comparing your data against other diseases, if you have time for it.


**Workshop notebooks:** **01 — Matching & Loading (you are here)**&nbsp;·&nbsp;[02 — Overlay & Enrichment](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/02_overlay_enrichment.ipynb)&nbsp;·&nbsp;[03 — Bridging](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/03_bridging.ipynb)&nbsp;·&nbsp;[04 — Disease Modules (optional)](https://colab.research.google.com/github/marlene-grabner/NetworkMedicine_Workshop/blob/main/04_disease_modules_optional.ipynb)